In [46]:
import os
import pandas as pd
import subprocess
from time import sleep

TARGET_CLASS_MID = "/m/025wky1"
SEGMENTS_FILE = "/Users/minimac/Desktop/fire_classification/unbalanced_train_segments.csv"
OUTPUT_DIR = "/Users/minimac/Desktop/fire_classification/airconditioner"
MAX_DOWNLOADS =30

print("시작")

# CSV 읽기
segments = pd.read_csv(
    SEGMENTS_FILE,
    sep=",",
    skipinitialspace=True,  # ← 각 필드 앞 공백 제거
    quotechar='"',          # ← "로 감싸진 문자열 안 쉼표 무시
    engine="python",        # ← 유연한 파싱
    dtype=str
)

segments["positive_labels"] = segments["positive_labels"].str.strip()


# 디버깅 출력
print("열 이름:", segments.columns.tolist())
print("데이터 일부:", segments.head())
print("🎉 모든 준비 완료!")


print("positive_labels 샘플 출력:")
print(segments["positive_labels"].head(10).tolist())


def includes_fire_alarm(label_str):
    label_str = label_str.strip().replace('"', '')  # 큰따옴표 제거
    labels = [x.strip() for x in label_str.split(',')]
    return TARGET_CLASS_MID in labels

fire_alarm_samples = segments[
    segments["positive_labels"].str.contains(
        fr"(^|,){TARGET_CLASS_MID}($|,)", na=False
    )
].reset_index(drop=True)
print("전체 샘플 수:", len(segments))
print("지정 샘플 수:", len(fire_alarm_samples))
print(fire_alarm_samples.head())



시작
열 이름: ['YTID', 'start_seconds', 'end_seconds', 'positive_labels']
데이터 일부:           YTID start_seconds end_seconds  \
0  ---1_cCGK4M         0.000      10.000   
1  ---2_BBVHAA        30.000      40.000   
2  ---B_v8ZoBY        30.000      40.000   
3  ---EDNidJUA        30.000      40.000   
4  ---N4cFAE1A        21.000      31.000   

                                     positive_labels  
0  /m/01g50p,/m/0284vy3,/m/06d_3,/m/07jdr,/m/07rwm0c  
1                                           /m/09x0r  
2                                           /m/04rlf  
3              /m/02qldy,/m/02zsn,/m/05zppz,/m/09x0r  
4                                  /m/04rlf,/m/09x0r  
🎉 모든 준비 완료!
positive_labels 샘플 출력:
['/m/01g50p,/m/0284vy3,/m/06d_3,/m/07jdr,/m/07rwm0c', '/m/09x0r', '/m/04rlf', '/m/02qldy,/m/02zsn,/m/05zppz,/m/09x0r', '/m/04rlf,/m/09x0r', '/m/019jd,/m/07yv9', '/m/04rlf,/m/0c1dj', '/m/012f08,/m/07yv9,/m/0k4j,/t/dd00134', '/m/04rlf', '/m/09x0r']


/var/folders/vz/1rv7tvd16l3315q95vmf3g3h0000gn/T/ipykernel_2041/4104824216.py:42: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  segments["positive_labels"].str.contains(


전체 샘플 수: 2041789
지정 샘플 수: 290
          YTID start_seconds end_seconds                 positive_labels
0  -AL_xXewmcg        20.000      30.000  /m/025wky1,/m/09x0r,/t/dd00125
1  -HqhVtG96S4       190.000     200.000             /m/025wky1,/m/09x0r
2  -XpHTpjNxbY       270.000     280.000             /m/025wky1,/m/09x0r
3  -Zfq_oF4REc        10.000      20.000                      /m/025wky1
4  0DzOKaK1jxo        12.000      22.000                      /m/025wky1


In [47]:


# 유튜브 오디오 다운로드 함수
def download_clip(youtube_id, start, end, output_path):
    duration = float(end) - float(start)
    command = [
        "yt-dlp",
        f"https://www.youtube.com/watch?v={youtube_id}",
        "--extract-audio",
        "--audio-format", "wav",
        "-o", output_path,
        "--quiet",
        "--no-warnings",
        "--postprocessor-args", f"-ss {start} -t {duration}"
    ]
    try:
        subprocess.run(command, check=True)
        return True
    except subprocess.CalledProcessError:
        return False

# 다운로드 루프
print("⏬ 다운로드 시작...")
success_count = 0

for i, row in fire_alarm_samples.iterrows():
    if success_count >= MAX_DOWNLOADS:
        break

    ytid = row["YTID"]
    start = row["start_seconds"]
    end = row["end_seconds"]
    output_filename = os.path.join(OUTPUT_DIR, f"{ytid}_{start}_{end}.wav")

    if os.path.exists(output_filename):
        print(f"[{i}] ⏭ 이미 존재: {ytid}")
        continue

    success = download_clip(ytid, start, end, output_filename)
    if success:
        success_count += 1
        print(f"[{success_count}] 다운로드 성공 : {ytid}")
    else:
        print(f"[{i}] XXXXXX : {ytid}")

    sleep(5)  # 유튜브 API 요청 제한 방지용 딜레이

print(f"\n🎉 다운로드 완료: 총 {success_count}개 성공")


⏬ 다운로드 시작...
[1] 다운로드 성공 : -AL_xXewmcg
[2] 다운로드 성공 : -HqhVtG96S4
[3] 다운로드 성공 : -XpHTpjNxbY
[4] 다운로드 성공 : -Zfq_oF4REc
[5] 다운로드 성공 : 0DzOKaK1jxo
[6] 다운로드 성공 : 0GqF4e_aA1c
[7] 다운로드 성공 : 0Jn20yU-jHE


ERROR: [youtube] 0PfRToJAdEc: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


[7] XXXXXX : 0PfRToJAdEc
[8] 다운로드 성공 : 0w5QcTqog3Q
[9] 다운로드 성공 : 131NNNWfpuE
[10] 다운로드 성공 : 1gr7Iw0ZZUg


ERROR: [youtube] 1wZpPH4CMe4: Video unavailable


[11] XXXXXX : 1wZpPH4CMe4
[11] 다운로드 성공 : 201bbKEo_jA


...

[21] XXXXXX : 4MRvq32wBzU
[19] 다운로드 성공 : 4katznyzimE
[20] 다운로드 성공 : 5MyuJia6Ho4
[21] 다운로드 성공 : 5QN3khGn9qo
[22] 다운로드 성공 : 5mAVgWb4ODg


ERROR: [youtube] 5rCgclQ1Qr0: Video unavailable


[26] XXXXXX : 5rCgclQ1Qr0
[23] 다운로드 성공 : 65ETCV_EeyU
[24] 다운로드 성공 : 6OsRHLiRWjE
[25] 다운로드 성공 : 6hdEtOJbRgU
[26] 다운로드 성공 : 6tMFsvo5YXw
[27] 다운로드 성공 : 6v7c1hiE4Zg
[28] 다운로드 성공 : 6vb-aJvM0PI
[29] 다운로드 성공 : 73HBpGQbBww


ERROR: [youtube] 7EYjHph6ZgM: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


[34] XXXXXX : 7EYjHph6ZgM
[30] 다운로드 성공 : 7iSCziLZgow

🎉 다운로드 완료: 총 30개 성공
